In [ ]:
from common import (
    run_random_split,
    run_categorical_split,
    run_continuous_split,
    run_a_split,
    get_high_hemoglobin_log,
    save_high_hemoglobin_pickle,
    save_high_hemoglobin_plot,
    explain_disparities,
    investigate_common_variants,
    SEED,
    NumericalColumn_Levenshtein_PermutationComparator
)
from pcomp.utils import enable_logging
import pandas as pd
from random import random

enable_logging()

In [ ]:
def save_results(comparator: NumericalColumn_Levenshtein_PermutationComparator, category: str):
    save_high_hemoglobin_pickle(comparator, category)
    save_high_hemoglobin_plot(comparator, category)

In [ ]:
print("Number of normal hemoglobin cases:", get_high_hemoglobin_log()["case:concept:name"].nunique())

# Random Split

- Make sure the approach isn't too sensitive

In [ ]:
random_split_comparator = run_random_split(log=get_high_hemoglobin_log())

save_results(random_split_comparator, f"random_split_{SEED}")

In [ ]:
explain_disparities(random_split_comparator)

# Split on Language

In [ ]:
language_comparator = run_categorical_split(
    "language",
    "ENGLISH",
    "?",
    log=get_high_hemoglobin_log()
)

save_results(language_comparator, "english_proficiency")

In [ ]:
explain_disparities(language_comparator, "English", "      ?")

# Split on Gender

In [ ]:
gender_comparator = run_categorical_split("gender", "M", "F", log=get_high_hemoglobin_log())
save_results(gender_comparator, "gender")

In [ ]:
explain_disparities(gender_comparator, "M", "F")

In [ ]:
investigate_common_variants(gender_comparator, "M", "F", 10)

# Race

In [ ]:
race_comparator = run_categorical_split("race", "WHITE", "BLACK/AFRICAN AMERICAN", log=get_high_hemoglobin_log())
save_results(race_comparator, "race/black_white")

In [ ]:
explain_disparities(race_comparator, "White", "Black")

In [ ]:
investigate_common_variants(race_comparator, "White", "Black", 5)

# Generalized Race
- Because of the size of the dataset, the number of patients quickly decreases. Try to counteract this by generalizing
some races

In [ ]:
# Only count each case once
def get_counts(log: pd.DataFrame, column: str):
    return log.drop_duplicates(subset=["case:concept:name"])[column].value_counts(dropna=False)

def generalize_race_label(race: str) -> str:
    return race.split("-")[0].strip().split("/")[0].strip()

In [ ]:
get_counts(get_high_hemoglobin_log(), "race").to_frame()

In [ ]:
log = get_high_hemoglobin_log()
log["generalized_race"] = log.apply(lambda row: generalize_race_label(row["race"]), axis=1)
get_counts(log, "generalized_race").to_frame()

## White vs Black

In [ ]:
wb_general_race_comparator = run_categorical_split("generalized_race", "WHITE", "BLACK", log=log)
save_results(wb_general_race_comparator, "general_race/white_black")

In [ ]:
explain_disparities(wb_general_race_comparator, "White", "Black")

## Balanced Comparison

In [ ]:
log_1 = log[log["generalized_race"] == "WHITE"]
log_2 = log[log["generalized_race"] == "BLACK"]

log_1_caseids = sorted(log_1["case:concept:name"].unique().tolist(), key=lambda x: random())
sampled_caseids = log_1_caseids[: log_2["case:concept:name"].nunique()]
log_1 = log_1[log_1["case:concept:name"].isin(sampled_caseids)]

In [ ]:
balanced_general_race_comparator = run_a_split(log_1, log_2)
save_results(balanced_general_race_comparator, "general_race/white_black_balanced")

In [ ]:
explain_disparities(balanced_general_race_comparator, "White", "Black")

## Black vs Asian

In [ ]:
ba_general_race_comparator = run_categorical_split("generalized_race", "BLACK", "ASIAN", log=log)
save_results(ba_general_race_comparator, "general_race/black_asian")

In [ ]:
explain_disparities(ba_general_race_comparator, "Black", "Asian")

## White vs Asian

In [ ]:
wa_general_race_comparator = run_categorical_split("generalized_race", "WHITE", "ASIAN", log=log)
save_results(wa_general_race_comparator, "general_race/white_asian")

In [ ]:
explain_disparities(wa_general_race_comparator, "White", "Asian")

# Age

In [ ]:
age_comparator = run_continuous_split("anchor_age", 70, log=get_high_hemoglobin_log())
save_results(age_comparator, "age")

In [ ]:
explain_disparities(age_comparator, "Age  < 70", "Age >= 70")

## Larger Gap

In [ ]:
gap_age_comparator = run_continuous_split("anchor_age", 60, 70, log=get_high_hemoglobin_log())
save_results(gap_age_comparator, "age_large_gap")

In [ ]:
explain_disparities(gap_age_comparator, " < 60", ">= 70")

# Insurance

## Medicare vs Medicaid

In [ ]:
medicare_medicaid_comparator = run_categorical_split("insurance", "Medicare", "Medicaid", log=get_high_hemoglobin_log())
save_results(medicare_medicaid_comparator, "insurance/medicare_medicaid")

In [ ]:
explain_disparities(medicare_medicaid_comparator, "Medicare", "Medicaid")

## Medicare vs Other

In [ ]:
medicare_other_comparator = run_categorical_split("insurance", "Medicare", "Other", log=get_high_hemoglobin_log())
save_results(medicare_other_comparator, "insurance/medicare_other")

In [ ]:
explain_disparities(medicare_other_comparator, "Medicare", "   Other")

## Medicaid vs Other

In [ ]:
medicaid_other_comparator = run_categorical_split("insurance", "Medicaid", "Other", log=get_high_hemoglobin_log())
save_results(medicaid_other_comparator, "insurance/medicaid_other")

In [ ]:
explain_disparities(medicaid_other_comparator, "Medicaid", "Other")

# Other vs Other (Sanity Check)

In [ ]:
other_vs_other_log = get_high_hemoglobin_log()
other_vs_other_log = other_vs_other_log[other_vs_other_log["insurance"] == "Other"]

other_vs_other_comparator = run_random_split(other_vs_other_log)
save_results(other_vs_other_comparator, "insurance/other_other")

In [ ]:
explain_disparities(other_vs_other_comparator)

# Marital Status
- At first thought, this should have no effect on the outcome
- But: Marital status encodes, to a certain extent, the age of the patient
- And we have shown that there are differences based on age

In [ ]:
get_counts(get_high_hemoglobin_log(), "marital_status").to_frame()

## Married vs Single

In [ ]:
married_single_comparator = run_categorical_split("marital_status", '"MARRIED"', '"SINGLE"', log=get_high_hemoglobin_log())
save_results(married_single_comparator, "marital_status/married_single")

In [ ]:
explain_disparities(married_single_comparator, "Married", " Single")

## Single vs Widowed
- Likely a large difference in age &rarr; Differences based on age

In [ ]:
single_widowed_comparator = run_categorical_split("marital_status", '"SINGLE"', '"WIDOWED"', log=get_high_hemoglobin_log())
save_results(single_widowed_comparator, "marital_status/single_widowed")

In [ ]:
explain_disparities(single_widowed_comparator, " Single", "Widowed")

## Married vs Widowed

In [ ]:
married_widowed_comparator = run_categorical_split("marital_status", '"MARRIED"', '"WIDOWED"', log=get_high_hemoglobin_log())
save_results(married_widowed_comparator, "marital_status/married_widowed")

In [ ]:
explain_disparities(single_widowed_comparator, "Married", "Widowed")